In [1]:
import os
import time
import pickle

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import kagglehub

# Data
dataset_path = kagglehub.dataset_download("harbhajansingh21/german-traffic-sign-dataset")

def load_pickle(file_name):
    with open(os.path.join(dataset_path, file_name), "rb") as file:
        return pickle.load(file)

def to_tensor(images):
    return torch.from_numpy(images.astype("float32") / 255.0).permute(0, 3, 1, 2)

train_data = load_pickle("train.p")
test_data = load_pickle("test.p")

X_train_tensor = to_tensor(train_data["features"])
X_test_tensor = to_tensor(test_data["features"])
y_test = test_data["labels"]

test_loader = DataLoader(
    TensorDataset(X_test_tensor, torch.from_numpy(y_test).long()),
    batch_size=64,
    shuffle=False,
)


# Model architecture (same as training notebook)
class TrafficSignCNNv2(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Load the final model on CPU
model_path = "traffic_sign_cnn_final.pt"
model = TrafficSignCNNv2()
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()


# Accuracy check
def pytorch_accuracy(model, data_loader):
    correct = 0
    with torch.no_grad():
        for images, labels in data_loader:
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
    return correct / len(data_loader.dataset)


print("Test images:", len(y_test))
print(f"PyTorch model test accuracy: {pytorch_accuracy(model, test_loader):.2%}")
print(f"Model file size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")

/Users/yigit/CodeProjects/deos-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/yigit/CodeProjects/deos-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Test images: 12630
PyTorch model test accuracy: 98.79%
Model file size: 2.59 MB


In [2]:
import onnx

onnx_path = "traffic_sign_cnn.onnx"
dummy_input = torch.randn(1, 3, 32, 32)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["image"],
    output_names=["scores"],
    dynamic_axes={"image": {0: "batch"}, "scores": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print("ONNX model is valid")
print(f"ONNX file size: {os.path.getsize(onnx_path) / (1024 * 1024):.2f} MB")
print("Input:", onnx_model.graph.input[0].name)
print("Output:", onnx_model.graph.output[0].name)

ONNX model is valid
ONNX file size: 2.58 MB
Input: image
Output: scores


/var/folders/hr/xzj4jyj177sgdh8h54tkf2540000gn/T/ipykernel_17564/3916497613.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


In [3]:
import onnxruntime as ort

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])


def onnx_accuracy(session, images, labels, batch_size=64):
    correct = 0
    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size].numpy()
        scores = session.run(["scores"], {"image": batch})[0]
        correct += (scores.argmax(axis=1) == labels[start:start + batch_size]).sum()
    return correct / len(images)


def measure_latency_ms(run_function, repeats=500):
    for _ in range(20):
        run_function()
    start_time = time.perf_counter()
    for _ in range(repeats):
        run_function()
    return (time.perf_counter() - start_time) / repeats * 1000


single_image_tensor = X_test_tensor[:1]
single_image_numpy = single_image_tensor.numpy()

with torch.inference_mode():
    pytorch_ms = measure_latency_ms(lambda: model(single_image_tensor))

onnx_ms = measure_latency_ms(lambda: session.run(["scores"], {"image": single_image_numpy}))

print(f"ONNX test accuracy: {onnx_accuracy(session, X_test_tensor, y_test):.2%}")
print()
print(f"PyTorch: {pytorch_ms:.3f} ms per image  (~{1000 / pytorch_ms:.0f} FPS)")
print(f"ONNX:    {onnx_ms:.3f} ms per image  (~{1000 / onnx_ms:.0f} FPS)")

ONNX test accuracy: 98.79%

PyTorch: 0.460 ms per image  (~2173 FPS)
ONNX:    0.408 ms per image  (~2451 FPS)


In [4]:
from onnxruntime.quantization import (
    CalibrationDataReader,
    QuantFormat,
    QuantType,
    quantize_static,
)


class TrafficSignCalibrationReader(CalibrationDataReader):
    def __init__(self, images, num_samples=500):
        rng = np.random.default_rng(0)
        indices = rng.choice(len(images), num_samples, replace=False)
        self.samples = [images[index:index + 1].numpy() for index in indices]
        self.position = 0

    def get_next(self):
        if self.position >= len(self.samples):
            return None
        sample = {"image": self.samples[self.position]}
        self.position += 1
        return sample


int8_path = "traffic_sign_cnn_int8.onnx"

quantize_static(
    model_input=onnx_path,
    model_output=int8_path,
    calibration_data_reader=TrafficSignCalibrationReader(X_train_tensor),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
)

print(f"FP32 ONNX size: {os.path.getsize(onnx_path) / (1024 * 1024):.2f} MB")
print(f"INT8 ONNX size: {os.path.getsize(int8_path) / (1024 * 1024):.2f} MB")

FP32 ONNX size: 2.58 MB
INT8 ONNX size: 0.66 MB


In [5]:
from torchvision.transforms import v2

hard_transform = v2.Compose([
    v2.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.4, contrast=0.4),
])

torch.manual_seed(0)
X_test_hard = torch.stack([hard_transform(image) for image in X_test_tensor])
hard_test_loader = DataLoader(
    TensorDataset(X_test_hard, torch.from_numpy(y_test).long()),
    batch_size=64,
    shuffle=False,
)

int8_session = ort.InferenceSession(int8_path, providers=["CPUExecutionProvider"])
int8_ms = measure_latency_ms(lambda: int8_session.run(["scores"], {"image": single_image_numpy}))


def size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)


results = [
    ("PyTorch FP32", size_mb(model_path),
     pytorch_accuracy(model, test_loader), pytorch_accuracy(model, hard_test_loader), pytorch_ms),
    ("ONNX FP32", size_mb(onnx_path),
     onnx_accuracy(session, X_test_tensor, y_test), onnx_accuracy(session, X_test_hard, y_test), onnx_ms),
    ("ONNX INT8", size_mb(int8_path),
     onnx_accuracy(int8_session, X_test_tensor, y_test), onnx_accuracy(int8_session, X_test_hard, y_test), int8_ms),
]

print(f"{'Model':<14} | {'Size':>8} | {'Normal test':>11} | {'Hard test':>9} | {'Latency':>10}")
print("-" * 66)
for name, size, normal_accuracy, hard_accuracy, latency in results:
    print(f"{name:<14} | {size:>5.2f} MB | {normal_accuracy:>11.2%} | {hard_accuracy:>9.2%} | {latency:>7.3f} ms")

Model          |     Size | Normal test | Hard test |    Latency
------------------------------------------------------------------
PyTorch FP32   |  2.59 MB |      98.79% |    97.53% |   0.460 ms
ONNX FP32      |  2.58 MB |      98.79% |    97.53% |   0.408 ms
ONNX INT8      |  0.66 MB |      98.80% |    97.57% |   0.152 ms
